# Accuracy Vergleich: CIFAR-10 (Float vs. KAN vs. Integer-KAN)

In [19]:
import os
import sys
import importlib
import inspect

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import converted_KAN
import converted_KAN.accuracy_cost as accuracy_cost

importlib.reload(accuracy_cost)

from converted_KAN import (
    convert_to_kan,
    convert_to_int_kan,
    IntKANWrapper,
)
from converted_KAN.ops_counter import count_ops

evaluate_accuracy = accuracy_cost.evaluate_accuracy

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [20]:
def eval_with_progress(model, loader, device):
    sig = inspect.signature(evaluate_accuracy)
    if "show_progress" in sig.parameters:
        return evaluate_accuracy(model, loader, device=device, show_progress=True)
    return evaluate_accuracy(model, loader, device=device)

## Modell laden (CIFAR10 ResNet20 aus torch.hub)

In [21]:
HUB_REPO = "chenyaofo/pytorch-cifar-models"
HUB_MODEL = "cifar10_resnet20"

model = torch.hub.load(HUB_REPO, HUB_MODEL, pretrained=True, verbose=True)
model = model.to(device).eval()
model

Using cache found in /home/janis/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

## CIFAR-10 Testset

In [22]:
BATCH_SIZE = 128

normalize = transforms.Normalize(
    mean=[0.4914, 0.4822, 0.4465],
    std=[0.2470, 0.2435, 0.2616],
)

transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
len(test_loader)

79

## Accuracy Vergleich

In [ ]:
# 1) Float-Modell
acc_float = eval_with_progress(model, test_loader, device)
print(f"Accuracy Float:     {acc_float:.4f}")
# 2) KAN (Float)
kan_model = convert_to_kan(model, inplace=False).to(device).eval()
acc_kan = eval_with_progress(kan_model, test_loader, device)
print(f"Accuracy KAN:       {acc_kan:.4f}")
# 3) Integer-KAN (fixed-point)
frac_bits = 8
int_model = convert_to_int_kan(model, frac_bits=frac_bits)
int_runner = IntKANWrapper(int_model, frac_bits=frac_bits, return_int=False).to(device).eval()
acc_int_kan = eval_with_progress(int_runner, test_loader, device)

print(f"Accuracy Int-KAN:   {acc_int_kan:.4f}")

## Ops Counter: KAN vs. Nicht-KAN (inkl. per-Layer)

In [ ]:
input_shape = (1, 3, 32, 32)

base_counts = count_ops(model, input_shape, device=device, per_layer=True)
kan_counts = count_ops(kan_model, input_shape, device=device, per_layer=True)

print("Non-KAN total:\n", base_counts["total"])
print("\nKAN total:\n", kan_counts["total"])

print("\nNon-KAN per-layer:\n", base_counts["per_layer"])
print("\nKAN per-layer:\n", kan_counts["per_layer"])